# Mage-VL-Cowork — computer-use agent on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/mage-vl-cowork-computer-use-eb5f/colab/Mage-VL-Cowork-computer-use-agent.ipynb)

[`microsoft/Mage-VL-Cowork`](https://huggingface.co/microsoft/Mage-VL-Cowork) is Microsoft's Apache-2.0 **4B vision-language computer-use agent**: one policy that either clicks a screenshot (`computer_use`, 11 GUI primitives in `[0, 999]` coordinates) or runs a shell command (`bash`). The backbone is [`Qwen/Qwen3-4B-Instruct-2507`](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) plus the Mage-ViT vision encoder (`model_type: llava_onevision2`). About **9.48 GB** in bfloat16 across two shards, **32,768** tokens of active context. Headline score: **66.0% OSWorld-Verified** (best among 4B models on the card).

**Why this model.** Most computer-use agents stay on pixels and burn tokens re-deriving a spreadsheet edit through dozens of clicks. Mage-VL-Cowork internalizes the routing decision — click when the work is on-screen, `bash` when the shell is faster and more verifiable — with no external orchestrator.

This notebook is a **safe single-step policy demo**, not live desktop control. It renders synthetic PIL windows, prints the predicted tool call, and draws any predicted click. Predicted `bash` is printed, not executed (a hardcoded allowlisted `wc` on a temp dir is the only command this notebook runs).

**Licence:** [Apache 2.0](https://www.apache.org/licenses/LICENSE-2.0). Custom Hub code (`configuration_llava_onevision2.py`, `modeling_llava_onevision2.py`) so every load uses `trust_remote_code=True`.

**Runtime / VRAM assumption:** *Runtime > Change runtime type > T4 GPU*. A free T4 has about **15 GB** of VRAM, **no native bfloat16 tensor cores**, and **no FlashAttention-2**. The card's offline snippet loads `torch.bfloat16` and the SGLang path uses FlashInfer + `SGLANG_MAX_PIXELS=4000000` for a `1920×1080` desktop. That recipe does not fit here. This notebook loads **float16 + SDPA** first (same 9.5 GB of weights as bf16), caps screenshots at **1280×720** / `max_pixels≈0.92M` (not the card's 4M), and keeps `max_new_tokens` at **640** (not 4096). If float16 plus the vision activations overflow ~15 GB, the load cell falls back to **4-bit NF4** via bitsandbytes (about 4.8 GB of packed weights). Expected peak: **~11–14 GB** in float16 or **~7–11 GB** in 4-bit — an estimate from the weight file plus one 720p image, not a measured Colab number yet. The last cell prints `torch.cuda.max_memory_allocated`.

**Not in this notebook:** SGLang / the `xiaoyizhang/mage-vl-cowork` Docker runtime, OSWorld rollouts, a real desktop or VNC session, FlashAttention-2, or the card's 50-step `MAGEHybridAgent` loop.

**Links:** [model card](https://huggingface.co/microsoft/Mage-VL-Cowork) · [GitHub `mage_vl_cowork`](https://github.com/microsoft/Mage/tree/main/mage_vl_cowork) · [project page](https://microsoft.github.io/Mage/cowork/) · [Mage-VL base](https://huggingface.co/microsoft/Mage-VL)


## 1. Install Transformers and bitsandbytes

The [model card](https://huggingface.co/microsoft/Mage-VL-Cowork) asks for `transformers>=4.57.3`, `accelerate`, `pillow`, `torch`, `torchvision`, and `qwen-vl-utils`, and says to always pass `trust_remote_code=True` (`llava_onevision2` is not a stock Qwen3-VL class). The checkpoint was saved with `transformers_version: 4.57.3`; the official SGLang container pins Transformers `5.7.0`. Either side of that range is fine.

Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled `torch`. Keep Colab's torch. Do **not** install `flash-attn`. `bitsandbytes` is only the 4-bit NF4 fallback if float16 overflows.


In [ ]:
%pip install -q "transformers>=4.57.3" "accelerate>=1.1.0" "bitsandbytes>=0.45.0" "qwen-vl-utils>=0.0.11" pillow

# Card: transformers>=4.57.3 + accelerate + pillow + qwen-vl-utils + trust_remote_code.
# bitsandbytes is the 4-bit NF4 fallback if float16 + a screenshot overflows ~15 GB.
# Use %pip, not python -I -m pip — isolated mode drops Colab site-packages.
# Do NOT replace Colab's torch. Do NOT install flash-attn.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.


## 2. Check the GPU and print T4 caveats

The next cell must see a CUDA GPU. It prints `torch.cuda.is_available()`, the device name, and free VRAM, then stops if you are on CPU.


In [ ]:
import gc

import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__) >= Version("4.57.3"), (
    f"transformers {transformers.__version__} is loaded; Mage-VL-Cowork needs 4.57.3+. "
    "Run the install cell, then Runtime > Restart session."
)

print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2 and no native bfloat16 tensor cores.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

DEVICE = "cuda"
major, minor = torch.cuda.get_device_capability()
try:
    NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    NATIVE_BF16 = major >= 8
free, total = torch.cuda.mem_get_info()
DTYPE = torch.float16  # card default is bfloat16; T4 has no native bf16

print("transformers", transformers.__version__, "| torch", torch.__version__)
print("device:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"VRAM free: {free / 1024**3:.1f} GB of {total / 1024**3:.1f} GB")
print("native bf16:", NATIVE_BF16, "| FlashAttention-2: off (sdpa / eager only)")
print("load path: float16 first, 4-bit NF4 fallback if VRAM overflows")
print("screenshot cap: 1280x720 / max_pixels=921600 (card eval is 1920x1080 / 4e6)")


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB | "
        f"peak: {peak:.2f} GB | free: {free_now / 1024**3:.1f}/{total_now / 1024**3:.1f} GB{suffix}"
    )


## 3. Load Mage-VL-Cowork

`AutoProcessor` + `AutoModelForCausalLM` with `trust_remote_code=True`, matching the [card's offline Transformers example](https://huggingface.co/microsoft/Mage-VL-Cowork#4-offline-transformers-inference-example) and [`MAGEHybridAgent`](https://github.com/microsoft/Mage/tree/main/mage_vl_cowork). The processor class is `Qwen2_5_VLProcessor`; the LM is the bundled `LlavaOnevision2ForConditionalGeneration`. The first run downloads about **9.5 GB**.

The system prompt below is the verbatim `MAGE_HYBRID_SYSTEM_PROMPT` from the card / `mage_hybrid_agent.py`. That is where the `computer_use` + `bash` schemas and the `<think>` / `Action:` / `<tool_call>` contract live — do not skip it, and do not also pass a `tools=` list to the chat template (that would duplicate the schema). Official OSWorld sampling is `temperature=0.7`, `top_p=0.8`, `max_new_tokens=4096`, `enable_thinking=True`. Demos here stay **greedy** and **640** new tokens so a T4 finishes.


In [ ]:
import json
import re
from pathlib import Path

from PIL import Image, ImageDraw, ImageFont
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "microsoft/Mage-VL-Cowork"
SCREEN_W, SCREEN_H = 1280, 720
MAX_PIXELS = SCREEN_W * SCREEN_H  # 921_600; card eval uses 4_000_000
MAX_NEW_TOKENS = 640
LOAD_MODE = None
ATTN = None

# Verbatim MAGE_HYBRID_SYSTEM_PROMPT from the model card / mage_hybrid_agent.py.
MAGE_HYBRID_SYSTEM_PROMPT = """You are a multi-purpose intelligent assistant. Based on my requests, you can use tools to help me complete various tasks.

# Tools

You have access to the following functions:

<tools>
{"type": "function", "function": {"name": "computer_use", "description": "Use a mouse and keyboard to interact with a computer screen.", "parameters": {"properties": {"action": {"description": "* `click`: Click at (x,y). Use `button` for left/right/middle.\n* `double_click`: Double click at (x,y).\n* `triple_click`: Triple click at (x,y) to select line/paragraph.\n* `move`: Move cursor to (x,y) without clicking.\n* `drag`: Drag from `start` to `end` coordinates.\n* `type`: Type text string.\n* `keypress`: Press key(s). Single key or combo.\n* `scroll`: Scroll at optional (x,y) position. scroll_x/scroll_y in [-999,999], |999|=full screen. Negative scroll_y=down, negative scroll_x=left.\n* `wait`: Wait specified seconds.\n* `screenshot`: Capture the current screen without changing anything.\n* `terminate`: End the task.", "enum": ["click", "double_click", "triple_click", "move", "drag", "type", "keypress", "scroll", "wait", "screenshot", "terminate"], "type": "string"}, "coordinate": {"description": "(x,y) in [0,999].", "type": "array"}, "button": {"description": "Mouse button for click: left/right/middle.", "type": "string", "enum": ["left", "right", "middle"]}, "start": {"description": "(x,y) drag start in [0,999].", "type": "array"}, "end": {"description": "(x,y) drag end in [0,999].", "type": "array"}, "keys": {"description": "Key list for keypress.", "type": "array"}, "text": {"description": "Text for action=type.", "type": "string"}, "scroll_x": {"description": "Horizontal scroll [-999,999]. Negative=left.", "type": "number"}, "scroll_y": {"description": "Vertical scroll [-999,999]. Negative=down.", "type": "number"}, "time": {"description": "Seconds for wait.", "type": "number"}, "status": {"description": "success/failure for terminate.", "type": "string", "enum": ["success", "failure"]}, "msg_to_user": {"description": "Optional message for terminate.", "type": "string"}}, "required": ["action"], "type": "object"}}}
{"type": "function", "function": {"name": "bash", "description": "Run a shell command on the machine and read its output. Use it for file inspection and edits, and for anything the shell does more reliably than the GUI; the result comes back as text, not as a screenshot.", "parameters": {"properties": {"command": {"description": "The shell command to run.", "type": "string"}}, "required": ["command"], "type": "object"}}}
</tools>

For every step, reply in EXACTLY this format with NO suffix:

<think>
your reasoning paragraph describing the current screen state, how it connects to the task goal, and the rationale for the chosen action
</think>

Action: one short imperative sentence describing the next move.

<tool_call>
{"name": "example_function_name", "arguments": {"example_parameter_1": "value_1"}}
</tool_call>

<IMPORTANT>
Reminder:
- Function calls MUST follow the specified format
- Required parameters MUST be specified
- Reasoning goes inside <think>...</think> BEFORE the Action line; nothing after the last tool_call
- You may output at most 15 <tool_call> blocks per response
- Use `type` for text input, `keypress` for shortcuts and special keys
- Every tool_call in one response must use the same function
</IMPORTANT>

# Response format

For every step:
1) <think>...</think> block with your reasoning.
2) Action: a short imperative sentence.
3) One or more <tool_call>...</tool_call> blocks (max 15).

Rules:
- The screen's resolution is 1000x1000. Coordinates are in [0, 999] range.
- Scroll amounts are normalized: |999| = one full screen width/height.
- For scroll, you may include coordinate to scroll at that position.
- For drag, specify start and end coordinates.
- Choose `bash` when the shell is the reliable way to do or check something, and `computer_use` when the work has to happen in the interface on screen.
- A `bash` call is answered with the command's text output; a `computer_use` call is answered with a new screenshot.
- Be brief: one sentence for Action.
- Do not output anything after the last tool call.
- If finishing, use action=terminate."""


def _load_mage(four_bit=False, attn=None):
    kwargs = {
        "trust_remote_code": True,
        "device_map": "auto",
        "low_cpu_mem_usage": True,
    }
    if attn:
        kwargs["attn_implementation"] = attn
    if four_bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
    else:
        kwargs["torch_dtype"] = DTYPE
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs).eval()


def reload_four_bit(reason):
    global model, LOAD_MODE, ATTN
    print(reason)
    if "model" in globals() and model is not None:
        del model
    gc.collect()
    torch.cuda.empty_cache()
    last_error = None
    loaded = None
    used_attn = None
    for attn in ("sdpa", "eager", None):
        try:
            loaded = _load_mage(four_bit=True, attn=attn)
            used_attn = attn or "default"
            break
        except Exception as exc:
            last_error = exc
            print(f"4-bit load failed (attn={attn!r}):", type(exc).__name__, exc)
            if loaded is not None:
                del loaded
                loaded = None
            gc.collect()
            torch.cuda.empty_cache()
    if loaded is None:
        raise RuntimeError("Could not load Mage-VL-Cowork in 4-bit NF4.") from last_error
    model = loaded
    LOAD_MODE = "4-bit"
    ATTN = used_attn
    print("loaded 4-bit NF4 | attention:", ATTN)
    print_vram("after 4-bit reload")


print("Loading processor (trust_remote_code; Qwen2_5_VLProcessor)...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
if hasattr(processor, "image_processor"):
    processor.image_processor.max_pixels = MAX_PIXELS
    processor.image_processor.size = {"longest_edge": MAX_PIXELS, "shortest_edge": 3136}

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(f"Loading {MODEL_ID} in float16 (first run ~9.5 GB)...")
model = None
last_error = None
for attn in ("sdpa", "eager", None):
    try:
        model = _load_mage(four_bit=False, attn=attn)
        ATTN = attn or "default"
        LOAD_MODE = "float16"
        break
    except Exception as exc:
        last_error = exc
        print(f"float16 load failed (attn={attn!r}):", type(exc).__name__, exc)
        if model is not None:
            del model
            model = None
        gc.collect()
        torch.cuda.empty_cache()

if model is None:
    reload_four_bit(f"float16 load failed, retrying 4-bit: {last_error}")

n_params = sum(p.numel() for p in model.parameters()) / 1e9
print("class:", type(model).__name__)
print(f"parameters: {n_params:.2f}B | load: {LOAD_MODE} | attention: {ATTN}")
print("dtype:", getattr(model, "dtype", None))
print_vram("after load")


def render_prompt(messages, enable_thinking=True):
    """Card + MAGEHybridAgent: apply_chat_template(..., enable_thinking=True)."""
    attempts = [
        {"tokenize": False, "add_generation_prompt": True, "chat_template_kwargs": {"enable_thinking": enable_thinking}},
        {"tokenize": False, "add_generation_prompt": True, "enable_thinking": enable_thinking},
        {"tokenize": False, "add_generation_prompt": True},
    ]
    last_exc = None
    for kwargs in attempts:
        try:
            text = processor.apply_chat_template(messages, **kwargs)
            if isinstance(text, list):
                text = text[0] if text else ""
            if isinstance(text, str) and text:
                return text
        except TypeError as exc:
            last_exc = exc
    if last_exc is not None:
        raise last_exc
    raise RuntimeError("apply_chat_template failed")


def move_inputs(inputs):
    device = next(model.parameters()).device
    dtype = getattr(model, "dtype", DTYPE)
    moved = {}
    for key, value in inputs.items():
        if not hasattr(value, "to"):
            moved[key] = value
            continue
        if key == "pixel_values":
            moved[key] = value.to(device=device, dtype=dtype)
        else:
            moved[key] = value.to(device)
    return moved


def generate_ids(inputs, max_new_tokens):
    with torch.inference_mode():
        return model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # greedy for a stable T4 demo; card 66.0% uses 0.7 / 0.8
            temperature=None,
            top_p=None,
        )


def parse_tool_calls(text):
    calls = []
    for match in re.finditer(r"<tool_call>\s*(.*?)\s*</tool_call>", text, flags=re.DOTALL):
        raw = match.group(1).strip()
        try:
            calls.append(json.loads(raw))
        except json.JSONDecodeError:
            calls.append({"raw": raw})
    return calls


def extract_action_line(text):
    match = re.search(r"^Action:\s*(.+)$", text, flags=re.MULTILINE)
    return match.group(1).strip() if match else ""


def extract_think(text):
    match = re.search(r"<think>(.*?)</think>", text, flags=re.DOTALL)
    return match.group(1).strip() if match else ""


def to_pixels(coord, width, height):
    """Card: x = round(coord_x * width / 1000), coords in [0, 999]."""
    x, y = float(coord[0]), float(coord[1])
    return int(round(x * width / 1000.0)), int(round(y * height / 1000.0))


def policy_step(task, image, max_new_tokens=MAX_NEW_TOKENS):
    """One hybrid-policy turn. Does not execute computer_use or bash."""
    global model
    messages = [
        {"role": "system", "content": MAGE_HYBRID_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": f"Task: {task}"},
            ],
        },
    ]
    prompt = render_prompt(messages, enable_thinking=True)
    inputs = processor(text=[prompt], images=[image.convert("RGB")], return_tensors="pt")
    inputs = move_inputs(inputs)
    try:
        output_ids = generate_ids(inputs, max_new_tokens)
    except RuntimeError as exc:
        overflow = any(token in str(exc).lower() for token in ("out of memory", "overflow", "nan", "inf"))
        if LOAD_MODE == "4-bit" or not overflow:
            raise
        reload_four_bit(f"float16 generate overflowed, reloading 4-bit: {exc}")
        inputs = processor(text=[prompt], images=[image.convert("RGB")], return_tensors="pt")
        inputs = move_inputs(inputs)
        output_ids = generate_ids(inputs, max_new_tokens)
    prompt_len = inputs["input_ids"].shape[-1]
    text = processor.tokenizer.decode(output_ids[0, prompt_len:], skip_special_tokens=True)
    return {
        "text": text,
        "think": extract_think(text),
        "action": extract_action_line(text),
        "calls": parse_tool_calls(text),
        "prompt_tokens": prompt_len,
        "new_tokens": int(output_ids.shape[-1] - prompt_len),
    }


def _font(size=18):
    for path in (
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
    ):
        if Path(path).exists():
            return ImageFont.truetype(path, size)
    return ImageFont.load_default()


def _text_size(draw, text, font):
    box = draw.textbbox((0, 0), text, font=font)
    return box[2] - box[0], box[3] - box[1]


def draw_window(canvas, box, title, fill=(246, 247, 249)):
    x0, y0, x1, y1 = box
    draw = ImageDraw.Draw(canvas)
    draw.rounded_rectangle(box, radius=10, fill=fill, outline=(60, 64, 72), width=2)
    draw.rectangle((x0, y0, x1, y0 + 36), fill=(47, 52, 61))
    draw.ellipse((x0 + 12, y0 + 12, x0 + 24, y0 + 24), fill=(232, 80, 80))
    draw.ellipse((x0 + 30, y0 + 12, x0 + 42, y0 + 24), fill=(240, 190, 60))
    draw.ellipse((x0 + 48, y0 + 12, x0 + 60, y0 + 24), fill=(80, 180, 90))
    draw.text((x0 + 72, y0 + 8), title, fill=(240, 242, 245), font=_font(16))
    return canvas


def render_contact_form(width=SCREEN_W, height=SCREEN_H):
    """Fake Acme Portal contact form. Submit button is large and high-contrast."""
    img = Image.new("RGB", (width, height), (38, 70, 110))
    draw = ImageDraw.Draw(img)
    draw.rectangle((0, height - 40, width, height), fill=(32, 36, 42))
    draw.text((16, height - 30), "Activities    Files    Firefox", fill=(220, 224, 230), font=_font(15))
    window = (180, 70, 1100, 640)
    draw_window(img, window, "Acme Portal — Contact")
    x0, y0, x1, y1 = window
    draw.text((x0 + 36, y0 + 56), "New ticket", fill=(30, 34, 40), font=_font(28))
    draw.text((x0 + 36, y0 + 96), "Fill the form, then click Submit.", fill=(80, 86, 96), font=_font(16))

    def field(label, value, top):
        draw.text((x0 + 36, top), label, fill=(50, 54, 62), font=_font(15))
        draw.rounded_rectangle((x0 + 36, top + 24, x1 - 36, top + 64), radius=6, fill=(255, 255, 255), outline=(170, 176, 186), width=2)
        draw.text((x0 + 48, top + 34), value, fill=(40, 44, 52), font=_font(16))

    field("Name", "Ada Lovelace", y0 + 140)
    field("Email", "ada@example.com", y0 + 220)
    draw.text((x0 + 36, y0 + 300), "Message", fill=(50, 54, 62), font=_font(15))
    draw.rounded_rectangle((x0 + 36, y0 + 324, x1 - 36, y0 + 430), radius=6, fill=(255, 255, 255), outline=(170, 176, 186), width=2)
    draw.text((x0 + 48, y0 + 340), "Please provision the staging VM.", fill=(40, 44, 52), font=_font(16))

    cancel = (x1 - 360, y1 - 90, x1 - 210, y1 - 42)
    submit = (x1 - 190, y1 - 90, x1 - 36, y1 - 42)
    draw.rounded_rectangle(cancel, radius=8, fill=(226, 229, 234), outline=(140, 146, 156), width=2)
    draw.rounded_rectangle(submit, radius=8, fill=(26, 115, 232), outline=(16, 80, 170), width=2)
    cw, ch = _text_size(draw, "Cancel", _font(18))
    sw, sh = _text_size(draw, "Submit", _font(18))
    draw.text(((cancel[0] + cancel[2] - cw) / 2, (cancel[1] + cancel[3] - ch) / 2), "Cancel", fill=(50, 54, 62), font=_font(18))
    draw.text(((submit[0] + submit[2] - sw) / 2, (submit[1] + submit[3] - sh) / 2), "Submit", fill=(255, 255, 255), font=_font(18))
    return img, {"submit": submit, "cancel": cancel, "window": window}


def render_hybrid_desktop(width=SCREEN_W, height=SCREEN_H):
    """Desktop with a file manager (bash-shaped work) and a small web form (GUI)."""
    img = Image.new("RGB", (width, height), (52, 92, 64))
    draw = ImageDraw.Draw(img)
    draw.rectangle((0, height - 40, width, height), fill=(32, 36, 42))
    draw.text((16, height - 30), "Activities    Files    Firefox    Terminal", fill=(220, 224, 230), font=_font(15))
    files = (40, 50, 520, 640)
    draw_window(img, files, "Home — /home/user")
    fx0, fy0, fx1, fy1 = files
    rows = [
        ("data/", "folder", "3 items"),
        ("budget.xlsx", "spreadsheet", "18 KB"),
        ("notes.txt", "text", "2 KB"),
        ("invoice.pdf", "document", "94 KB"),
    ]
    for i, (name, kind, size) in enumerate(rows):
        top = fy0 + 56 + i * 44
        draw.rectangle((fx0 + 16, top, fx1 - 16, top + 40), fill=(236, 238, 242) if i % 2 == 0 else (246, 247, 249))
        draw.text((fx0 + 28, top + 10), f"{name:<16}  {kind:<12}  {size}", fill=(30, 34, 40), font=_font(15))

    form = (560, 80, 1240, 620)
    draw_window(img, form, "Firefox — Acme Portal")
    gx0, gy0, gx1, gy1 = form
    draw.text((gx0 + 28, gy0 + 56), "Contact", fill=(30, 34, 40), font=_font(24))
    draw.rounded_rectangle((gx0 + 28, gy0 + 110, gx1 - 28, gy0 + 160), radius=6, fill="white", outline=(170, 176, 186), width=2)
    draw.text((gx0 + 40, gy0 + 124), "Name: Ada Lovelace", fill=(40, 44, 52), font=_font(16))
    draw.rounded_rectangle((gx0 + 28, gy0 + 180, gx1 - 28, gy0 + 230), radius=6, fill="white", outline=(170, 176, 186), width=2)
    draw.text((gx0 + 40, gy0 + 194), "Email: ada@example.com", fill=(40, 44, 52), font=_font(16))
    submit = (gx1 - 200, gy1 - 100, gx1 - 36, gy1 - 48)
    draw.rounded_rectangle(submit, radius=8, fill=(26, 115, 232), outline=(16, 80, 170), width=2)
    sw, sh = _text_size(draw, "Submit", _font(18))
    draw.text(((submit[0] + submit[2] - sw) / 2, (submit[1] + submit[3] - sh) / 2), "Submit", fill="white", font=_font(18))
    return img, {"submit": submit, "files": files, "form": form}


def draw_click(image, coord, color=(220, 40, 40)):
    img = image.copy()
    draw = ImageDraw.Draw(img)
    x, y = to_pixels(coord, *img.size)
    r = 18
    draw.ellipse((x - r, y - r, x + r, y + r), outline=color, width=4)
    draw.line((x - 28, y, x + 28, y), fill=color, width=3)
    draw.line((x, y - 28, x, y + 28), fill=color, width=3)
    label = f"pred click [{coord[0]}, {coord[1]}] → px ({x}, {y})"
    font = _font(16)
    tw, th = _text_size(draw, label, font)
    pad = 6
    lx, ly = min(max(x + 24, 8), img.size[0] - tw - 16), min(max(y - 36, 8), img.size[1] - th - 16)
    draw.rectangle((lx - pad, ly - pad, lx + tw + pad, ly + th + pad), fill=(20, 20, 20))
    draw.text((lx, ly), label, fill=(255, 230, 230), font=font)
    return img, (x, y)


def inside(box, xy):
    return box[0] <= xy[0] <= box[2] and box[1] <= xy[1] <= box[3]


def show_prediction(result, image=None, hit_boxes=None):
    print("----- raw policy output -----")
    print(result["text"])
    print("----- parsed -----")
    print("Action:", result["action"] or "(none)")
    print("tool calls:", json.dumps(result["calls"], indent=2))
    print(f"tokens: prompt {result['prompt_tokens']} + new {result['new_tokens']}")
    names = [call.get("name") for call in result["calls"]]
    print("channel:", names[0] if names else "(no tool_call)")
    if image is None:
        return result
    for call in result["calls"]:
        args = call.get("arguments") or {}
        coord = args.get("coordinate")
        if call.get("name") == "computer_use" and isinstance(coord, (list, tuple)) and len(coord) >= 2:
            annotated, xy = draw_click(image, coord)
            print(f"click pixels: {xy}  (card: x = round(coord * width / 1000))")
            if hit_boxes:
                for label, box in hit_boxes.items():
                    print(f"  inside {label}? {inside(box, xy)}  box={box}")
            display(annotated)
            return result
    display(image)
    return result


## 4. Demo 1 — click Submit on a synthetic form

Safe single-step GUI demo. The screenshot is a PIL fake of a contact form, not a real desktop. The task is **click the Submit button**. The cell prints the `computer_use` call (normalized `[0, 999]` coordinates) and draws the predicted click. Nothing is sent to a window manager.


In [ ]:
form_image, form_boxes = render_contact_form()
print("synthetic form", form_image.size, "| Submit box (pixels):", form_boxes["submit"])
display(form_image)

gui_task = "Click the Submit button on the contact form."
gui_result = policy_step(gui_task, form_image)
show_prediction(gui_result, image=form_image, hit_boxes={"Submit": form_boxes["submit"], "Cancel": form_boxes["cancel"]})
print_vram("after GUI demo")


## 5. Demo 2 — terminal channel (print the bash call, do not run it)

Ask the policy to count lines in every `.csv` under `~/data`. A desktop screenshot is still passed because the model is trained on screenshot-conditioned turns; the interesting output is whether it chooses **`bash`**.

The predicted command is **printed only**. This notebook never `os.system`s model text. A separate, hardcoded `wc -l` on a temp dir with three tiny CSVs shows what a sandboxed allowlisted equivalent would return.


In [ ]:
import subprocess
import tempfile

desktop_image, desktop_boxes = render_hybrid_desktop()
print("synthetic desktop", desktop_image.size)
display(desktop_image)

bash_task = (
    "Count the total number of lines in all .csv files under ~/data. "
    "Use the shell; do not click around in the file manager."
)
bash_result = policy_step(bash_task, desktop_image)
show_prediction(bash_result, image=desktop_image)

print()
print("----- model bash (NOT executed) -----")
for call in bash_result["calls"]:
    if call.get("name") == "bash":
        print(call.get("arguments", {}).get("command", ""))
    else:
        print("model chose", call.get("name"), "instead of bash")

sandbox = Path(tempfile.mkdtemp(prefix="mage_sandbox_"))
data_dir = sandbox / "data"
data_dir.mkdir()
(data_dir / "a.csv").write_text("id,name\n1,ada\n2,grace\n")
(data_dir / "b.csv").write_text("sku,qty\nA,2\n")
(data_dir / "readme.txt").write_text("not a csv\n")
allowlisted = ["wc", "-l", str(data_dir / "a.csv"), str(data_dir / "b.csv")]
print()
print("----- allowlisted sandbox only (hardcoded wc, not the model string) -----")
print("cwd:", sandbox)
print("cmd:", allowlisted)
print(subprocess.check_output(allowlisted, text=True))
print_vram("after bash demo")


## 6. Demo 3 — hybrid routing (GUI vs bash on the same desktop)

Same screenshot, two tasks. One is visual (click Submit on the Firefox form). The other is structured file work (merge A1:D1 in `/home/user/budget.xlsx` — the card's own example, which the policy should route to `bash` + `openpyxl` rather than clicking Calc).

Single-step only: we print the chosen channel for each task. No GUI action and no model-generated shell is executed.


In [ ]:
hybrid_tasks = [
    ("GUI-shaped", "Click the Submit button on the Acme Portal contact form."),
    (
        "bash-shaped",
        "Merge cells A1:D1 in /home/user/budget.xlsx, set the title to 'Q4 Summary', and save the file.",
    ),
]

routing_rows = []
for label, task in hybrid_tasks:
    print("=" * 72)
    print(label, "|", task)
    result = policy_step(task, desktop_image)
    show_prediction(
        result,
        image=desktop_image if label.startswith("GUI") else None,
        hit_boxes={"Submit": desktop_boxes["submit"]} if label.startswith("GUI") else None,
    )
    names = [call.get("name") for call in result["calls"]]
    routing_rows.append((label, names[0] if names else "(none)", result["action"]))
    print()

print("----- channel routing -----")
print(f"{'task':<12} {'tool':<16} action")
for label, tool, action in routing_rows:
    print(f"{label:<12} {tool:<16} {action}")
print_vram("after hybrid demo")
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")


## 7. Caveats and cleanup

**What this notebook is for.** An inference-only T4 demo of Mage-VL-Cowork: the official hybrid system prompt, `AutoProcessor.apply_chat_template` + `AutoModelForCausalLM.generate`, and three single-step policy samples (click, bash, channel routing). Use it to see the tool schema and the allocated / peak VRAM.

**Not a desktop agent.** Screenshots are PIL fakes. Predicted `computer_use` actions are drawn, not replayed. Predicted `bash` is printed. The only shell this notebook runs is a hardcoded `wc -l` on two files it created in a temp dir.

**Prompt contract (from the card).** Every turn is `<think>…</think>`, then `Action: …`, then one or more `<tool_call>{name, arguments}</tool_call>` blocks. Coordinates are normalized to `[0, 999]` regardless of the screenshot size (`x_px = round(coord_x * width / 1000)`). One response uses only one tool family.

**Thinking / sampling.** Official OSWorld-Verified 66.0% uses `enable_thinking=True`, `temperature=0.7`, `top_p=0.8`, `max_new_tokens=4096`, `history_n=3`, `images_to_keep=3`. This notebook is greedy and 640 new tokens so a free T4 finishes. Scores will not match the card.

**T4 / VRAM.** Float16 + SDPA first; 4-bit NF4 if load or generate overflows. Screenshots are 1280×720 with `max_pixels=921600`. Do not load bf16. Do not install FlashAttention-2. The 32k training window is not allocated. The last number below is `torch.cuda.max_memory_allocated` for this session.

**`trust_remote_code`.** `modeling_llava_onevision2.py` and `configuration_llava_onevision2.py` execute locally. Review them, and pin a revision if you need a frozen snapshot.

**Responsible use.** The card says to run this agent only in an isolated sandbox/VM, never against a personal desktop, credentials, or an open network. This notebook stays on synthetic images for that reason.

**Not covered here.** SGLang Docker (`xiaoyizhang/mage-vl-cowork:0.1.0`), the full `MAGEHybridAgent` OSWorld harness, multi-step working memory, or live web / desktop control.

**Licence.** Apache 2.0.

**Links again:** [model card](https://huggingface.co/microsoft/Mage-VL-Cowork) · [GitHub](https://github.com/microsoft/Mage/tree/main/mage_vl_cowork) · [project page](https://microsoft.github.io/Mage/cowork/)

Free the GPU before you open another notebook:


In [ ]:
print_vram("before cleanup")
try:
    del model
except NameError:
    pass
try:
    del processor
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print_vram("after cleanup")
    print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print("Mage-VL-Cowork released. Re-run the load cell if you want another prompt.")
